# Step 1: 에이전트 설계
### 1) 이름

PreReadAgent

### 2) 목적

영어 원서나 영어 학습 텍스트를 읽기 전에, 본문을 미리 분석해서 학습자가 더 쉽게 읽을 수 있도록 예습 자료를 만들어주는 에이전트입니다.

### 3) 해결하려는 문제

영어 원서를 처음 읽을 때 모르는 단어와 문장 때문에 진입장벽이 크다.
본문을 읽기 전에 핵심 단어와 핵심 내용을 미리 알면 이해가 쉬워진다.
예습 자료를 직접 정리하는 데 시간이 많이 든다.

### 4) 핵심 기능

입력된 영어 본문에서 핵심 단어를 추출한다.
본문의 핵심 내용을 짧게 요약한다.
읽기 전에 확인할 수 있는 간단한 예습 노트를 생성한다.

### 5) 그래프 구조

이번 과제에서는 원서예습 서비스의 가장 기초적인 흐름만 구현합니다.

#### 노드 설명
- extract_vocab: 본문에서 핵심 단어 추출
- make_preread_note: 핵심 단어와 본문을 바탕으로 예습 노트 생성

#### graph


<img src="./graph-diagram.png" width="200">




# Step2. 기초 구축


이 과제에서는 영어 원서 읽기 전 예습을 돕는 PreReadAgent를 설계하고 구현했습니다.
LangGraph의 StateGraph를 사용해 커스텀 State를 정의했고,
extract_vocab 노드와 make_preread_note 노드를 연결하여
입력 본문에서 핵심 단어를 추출하고 간단한 예습 노트를 생성하는 기본 그래프를 만들었습니다.

In [11]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class PreReadState(TypedDict):
    passage: str
    vocab_list: list[str]
    summary: str
    preread_note: str

In [12]:
def extract_vocab(state: PreReadState) -> PreReadState:
    text = state["passage"]

    words = (
        text.replace(",", " ")
        .replace(".", " ")
        .replace("!", " ")
        .replace("?", " ")
        .split()
    )

    vocab = []
    seen = set()

    for word in words:
        w = word.strip().lower()
        if len(w) >= 6 and w.isalpha() and w not in seen:
            seen.add(w)
            vocab.append(w)

    return {
        "vocab_list": vocab[:5]
    }


def make_preread_note(state: PreReadState) -> PreReadState:
    passage = state["passage"]
    vocab_list = state.get("vocab_list", [])

    sentences = [s.strip() for s in passage.split(".") if s.strip()]
    if len(sentences) >= 2:
        summary = ". ".join(sentences[:2]) + "."
    elif len(sentences) == 1:
        summary = sentences[0] + "."
    else:
        summary = passage

    vocab_text = ", ".join(vocab_list) if vocab_list else "No key vocabulary found"

    preread_note = f"""
[PreRead Note]

1. 핵심 요약
{summary}

2. 핵심 단어
{vocab_text}

3. 읽기 전 체크 포인트
- 본문이 어떤 주제에 관한 글인지 생각해보기
- 위 핵심 단어의 뜻을 먼저 확인해보기
- 요약 내용을 참고해서 글의 전체 흐름 예상해보기
""".strip()

    return {
        "summary": summary,
        "preread_note": preread_note
    }

In [13]:
graph_builder = StateGraph(PreReadState)

graph_builder.add_node("extract_vocab", extract_vocab)
graph_builder.add_node("make_preread_note", make_preread_note)

graph_builder.add_edge(START, "extract_vocab")
graph_builder.add_edge("extract_vocab", "make_preread_note")
graph_builder.add_edge("make_preread_note", END)

graph = graph_builder.compile()

In [14]:
sample_input = {
    "passage": "When I heard Karpathy say this, I wanted to find out how. How does one person ship like a team of twenty? Peter Steinberger built OpenClaw — 247K GitHub stars — essentially solo with AI agents. The revolution is here. A single builder with the right tooling can move faster than a traditional team. I'm Garry Tan, President & CEO of Y Combinator. I've worked with thousands of startups — Coinbase, Instacart, Rippling — when they were one or two people in a garage. Before YC, I was one of the first eng/PM/designers at Palantir, cofounded Posterous (sold to Twitter), and built Bookface, YC's internal social network. gstack is my answer. I've been building products for twenty years, and right now I'm shipping more code than I ever have. In the last 60 days: 600,000+ lines of production code (35% tests), 10,000-20,000 lines per day, part-time, while running YC full-time. Here's my last /retro across 3 projects: 140,751 lines added, 362 commits, ~115k net LOC in one week.",    "vocab_list": [],
    "summary": "",
    "preread_note": "",
}

result = graph.invoke(sample_input)
result

{'passage': "When I heard Karpathy say this, I wanted to find out how. How does one person ship like a team of twenty? Peter Steinberger built OpenClaw — 247K GitHub stars — essentially solo with AI agents. The revolution is here. A single builder with the right tooling can move faster than a traditional team. I'm Garry Tan, President & CEO of Y Combinator. I've worked with thousands of startups — Coinbase, Instacart, Rippling — when they were one or two people in a garage. Before YC, I was one of the first eng/PM/designers at Palantir, cofounded Posterous (sold to Twitter), and built Bookface, YC's internal social network. gstack is my answer. I've been building products for twenty years, and right now I'm shipping more code than I ever have. In the last 60 days: 600,000+ lines of production code (35% tests), 10,000-20,000 lines per day, part-time, while running YC full-time. Here's my last /retro across 3 projects: 140,751 lines added, 362 commits, ~115k net LOC in one week.",
 'voca

In [15]:
print("[원문]")
print(result["passage"])
print()

print("[핵심 단어]")
print(result["vocab_list"])
print()

print("[요약]")
print(result["summary"])
print()

print("[예습 노트]")
print(result["preread_note"])

[원문]
When I heard Karpathy say this, I wanted to find out how. How does one person ship like a team of twenty? Peter Steinberger built OpenClaw — 247K GitHub stars — essentially solo with AI agents. The revolution is here. A single builder with the right tooling can move faster than a traditional team. I'm Garry Tan, President & CEO of Y Combinator. I've worked with thousands of startups — Coinbase, Instacart, Rippling — when they were one or two people in a garage. Before YC, I was one of the first eng/PM/designers at Palantir, cofounded Posterous (sold to Twitter), and built Bookface, YC's internal social network. gstack is my answer. I've been building products for twenty years, and right now I'm shipping more code than I ever have. In the last 60 days: 600,000+ lines of production code (35% tests), 10,000-20,000 lines per day, part-time, while running YC full-time. Here's my last /retro across 3 projects: 140,751 lines added, 362 commits, ~115k net LOC in one week.

[핵심 단어]
['karpa